<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB16_Autoencoders_and_Anomaly_Detection_Real_Fuel_Data_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB16 · Clase 16 — Autoencoders y detección de anomalías**

## Bloque 3: IA — Deep Learning (continuación)

Todas las redes de este bloque hasta ahora eran **supervisadas**: `una etiqueta conocida (mina/roca, tipo de combustible, consumo del mes siguiente) le decía a la red qué predecir`. Esta clase vuelve al **aprendizaje no supervisado** (terreno de `NB09`) con una herramienta de Deep Learning construida para ello: el **autoencoder**. Lo usamos para una de sus aplicaciones reales más comunes — la **detección de anomalías** — aplicada de nuevo a los datos reales de `ship_fuel_efficiency.csv` de `NB07`/`NB09`/`NB14`, esta vez planteando una pregunta nueva: ¿qué travesías parecen *inusuales*, y ese "inusual" coincide con algo que realmente podamos explicar?

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar qué aprende un autoencoder, y por qué forzar los datos a pasar por un cuello de botella estrecho es la idea clave.
- Explicar cómo el error de reconstrucción se convierte en una puntuación de anomalía, sin necesitar etiquetas.
- Entrenar un autoencoder real sobre datos navales reales y elegir un umbral de anomalía defendible.
- Interpretar las anomalías señaladas contrastándolas con campos categóricos conocidos-pero-no-usados, el mismo hábito de validación de la clase de clustering de `NB09`.
- Relacionar los autoencoders con el PCA (`NB09`) como dos caminos distintos hacia la misma idea de fondo: la compresión.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso, hoja de ruta de hoy | 5 min | Teoría |
| 2 | ¿Por qué Deep Learning no supervisado? Autoencoders como compresión | 10 min | Teoría |
| 3 | Arquitectura del autoencoder: codificador, cuello de botella, decodificador | 10 min | Teoría + Práctica |
| 4 | Del error de reconstrucción a una puntuación de anomalía | 10 min | Teoría |
| 5 | Dataset real: revisitando los datos de combustible con una pregunta nueva | 10 min | Práctica |
| 6 | Práctica: construir y entrenar el autoencoder | 30 min | Práctica |
| 7 | Elegir un umbral y señalar anomalías | 20 min | Práctica |
| 8 | Interpretar las anomalías frente a campos conocidos | 10 min | Práctica |
| 9 | Autoencoders frente a PCA, y más allá de la detección de anomalías | 10 min | Teoría + Práctica |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son una orientación aproximada, no un guion cerrado — no hay descansos programados. Si cubrimos todo con tiempo de sobra, la clase termina antes; eso puede pasar y está bien.

---

## 1. Repaso: dónde estamos

- **`NB11`–`NB13`**: clasificadores supervisados — tabulares, y después imágenes.
- **`NB14`**: previsión de secuencias con una LSTM real.
- **`NB15`**: transfer learning, extracción de características frente a fine-tuning.
- **`NB16`** (hoy): Deep Learning no supervisado — autoencoders, aplicados a detección de anomalías.

Queda una sesión más después de esta para cerrar el Bloque 3 (`NB17`).

---

## 2. ¿Por qué Deep Learning no supervisado? Autoencoders como compresión

`NB09` hizo aprendizaje no supervisado sin Deep Learning: K-Means encontró grupos, PCA encontró proyecciones de menor dimensión, y ambos funcionaron enteramente a partir de la estructura propia de los datos, sin necesitar etiquetas. Un **[autoencoder](https://en.wikipedia.org/wiki/Autoencoder)** aplica esa misma idea de "sin etiquetas" a una red neuronal: se entrena para reconstruir su propia entrada lo más fielmente posible, después de exprimirla a través de una capa **cuello de botella** deliberadamente estrecha.

El truco está exactamente en ese cuello de botella. Si la red pudiera simplemente copiar la entrada a la salida, `no necesitaría ningún cuello de botella y no aprendería nada útil` — una tubería completamente abierta copia cualquier cosa. Forzar los datos a pasar por una capa más pequeña significa que la red solo puede reconstruir bien si aprende a **comprimir** la entrada en una representación compacta que aun así capture el patrón común a la mayoría de los datos. Cualquier cosa que la red no haya aprendido a representar bien en ese cuello de botella — por ser rara, inusual, o simplemente distinta de la mayoría de los ejemplos de entrenamiento — se reconstruye *mal*.

---

## 3. Arquitectura del autoencoder: codificador, cuello de botella, decodificador

Un autoencoder tiene dos mitades que comparten una capa central estrecha:

- El **codificador** (encoder) comprime la entrada hasta el cuello de botella.
- El **cuello de botella** es la representación comprimida — menos números que la entrada original.
- El **decodificador** (decoder) expande el cuello de botella de vuelta a la forma de la entrada original, intentando reconstruirla.

Estructuralmente, son solo dos MLP pequeños (terreno de `NB11`) pegados entre sí, entrenados de principio a fin para minimizar el **error de reconstrucción** — normalmente el error cuadrático medio entre la entrada y la salida, la misma métrica de regresión de `NB07` reutilizada para un objetivo completamente distinto:

Dibujemos esa forma de reloj de arena:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

layer_sizes = [5, 3, 2, 3, 5]
layer_x = [0, 1.6, 3.2, 4.8, 6.4]
labels = ["Input\n(features)", "Encoder\nhidden", "Bottleneck\n(compressed)", "Decoder\nhidden", "Output\n(reconstruction)"]
colors = ["lightblue", "lightcoral", "gold", "lightcoral", "lightblue"]

positions = []
for n, x in zip(layer_sizes, layer_x):
    ys = np.linspace(-(n - 1) / 2, (n - 1) / 2, n)
    positions.append([(x, y) for y in ys])

fig, ax = plt.subplots(figsize=(10, 5))

for l in range(len(layer_sizes) - 1):
    for x1, y1 in positions[l]:
        for x2, y2 in positions[l + 1]:
            ax.plot([x1, x2], [y1, y2], color="lightgray", lw=0.7, zorder=1)

for layer, color, label, x in zip(positions, colors, labels, layer_x):
    for lx, ly in layer:
        ax.add_patch(patches.Circle((lx, ly), 0.15, facecolor=color, edgecolor="black", zorder=2))
    ax.text(x, -3, label, ha="center", fontsize=9)

ax.set_xlim(-1, 7.4)
ax.set_ylim(-3.6, 3)
ax.axis("off")
ax.set_title("Autoencoder: encoder -> bottleneck -> decoder")
plt.tight_layout()
plt.show()

El punto más estrecho (en dorado, aquí de 2 unidades) es donde se fuerza a que ocurra la compresión. Fíjate en que no hay un "objetivo" separado en la función de pérdida — la entrada *es* `el objetivo, que es precisamente lo que hace que esto sea no supervisado`: no hay ninguna columna `y` en todo el bucle de entrenamiento de hoy.

---

## 4. Del error de reconstrucción a una puntuación de anomalía

Entrena el autoencoder con datos que son **mayoritariamente normales** (una suposición razonable para la mayoría de los datos operativos — la mayoría de las travesías no tienen nada de particular; las genuinamente anómalas son raras por definición). El cuello de botella de la red aprende a representar el patrón común a la mayoría de los ejemplos. Después, pasa *cualquier* ejemplo por la red entrenada y mide cuánto se aleja la reconstrucción del original — el **error de reconstrucción**:

$$
\text{error}(x) = \frac{1}{n} \lVert x - \text{decoder}(\text{encoder}(x)) \rVert^2
$$

Un ejemplo **típico** se reconstruye bien (error bajo) — la red ha visto en la práctica muchos ejemplos parecidos y ha aprendido a representar ese patrón. Un ejemplo **inusual** se reconstruye mal (error alto) — no encaja en la representación comprimida que la red aprendió, precisamente porque `la red nunca tuvo que aprender a representar bien los patrones raros para minimizar su pérdida media de entrenamiento`.

Esto nos da detección de anomalías con **cero anomalías etiquetadas necesarias** — una ventaja práctica real frente a los clasificadores supervisados de `NB08`/`NB11`–`NB13`, que todos necesitaban un ejemplo etiquetado de *cada* clase que pudieran reconocer. Un autoencoder solo necesita ejemplos de lo "normal".

---

## 5. Dataset real: revisitando los datos de combustible con una pregunta nueva

El mismo `ship_fuel_efficiency.csv` real de `NB07`/`NB09`/`NB14` — 1.440 travesías reales, 120 buques. `NB07` preguntaba "¿podemos predecir el consumo de combustible?"; `NB09` preguntaba "¿las travesías se agrupan en perfiles operativos?"; hoy planteamos una tercera pregunta, genuinamente distinta: **¿qué travesías individuales parecen operativamente inusuales, sin usar ninguna etiqueta?**

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
fuel.head()

Las mismas características numéricas libres de fuga que en `NB07`/`NB09` — `distance`, `fuel_consumption`, `engine_efficiency` — excluyendo todavía `CO2_emissions` por la misma razón de redundancia establecida en `NB07`:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np
import torch

feature_cols = ["distance", "fuel_consumption", "engine_efficiency"]
X = fuel[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val = train_test_split(X_scaled, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_full_t = torch.tensor(X_scaled, dtype=torch.float32)  # every voyage, for scoring later

X_train_t.shape

---

## 6. Práctica: construir y entrenar el autoencoder

3 características reales comprimidas hasta un cuello de botella de 2 unidades — suficiente para forzar una compresión real, lo bastante pequeño para entrenarse en segundos:

In [ ]:
import torch.nn as nn

class FuelAutoencoder(nn.Module):
    def __init__(self, n_features, bottleneck_size=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 8),
            nn.ReLU(),
            nn.Linear(8, bottleneck_size),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, 8),
            nn.ReLU(),
            nn.Linear(8, n_features),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

torch.manual_seed(42)
autoencoder = FuelAutoencoder(n_features=X_train_t.shape[1])
sum(p.numel() for p in autoencoder.parameters())

Entrena exactamente igual que cada red desde `NB11` — salvo que aquí la pérdida compara la salida del modelo con **su propia entrada**, no con una etiqueta separada:

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.01)

n_epochs = 100
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    autoencoder.train()
    optimizer.zero_grad()
    reconstruction = autoencoder(X_train_t)
    loss = criterion(reconstruction, X_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    autoencoder.eval()
    with torch.no_grad():
        val_reconstruction = autoencoder(X_val_t)
        val_loss = criterion(val_reconstruction, X_val_t)
    val_losses.append(val_loss.item())

plt.plot(train_losses, label="Training reconstruction loss")
plt.plot(val_losses, label="Validation reconstruction loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Autoencoder training")
plt.legend()
plt.show()

**Pruébalo tú mismo**: ver la curva de pérdida bajar es una cosa — mira lo que la red reconstruye realmente para unas cuantas travesías reales, en unidades reales (deshaciendo el escalado), para ver en qué se traduce concretamente "parecido al original".

In [ ]:
sample_idx = [0, 1, 2, 3, 4]
with torch.no_grad():
    reconstructed_sample = autoencoder(X_full_t[sample_idx]).numpy()

original_real = scaler.inverse_transform(X_full_t[sample_idx].numpy())
reconstructed_real = scaler.inverse_transform(reconstructed_sample)

comparison_df = pd.DataFrame({
    "distance (actual)": original_real[:, 0],
    "distance (reconstructed)": reconstructed_real[:, 0],
    "fuel_consumption (actual)": original_real[:, 1],
    "fuel_consumption (reconstructed)": reconstructed_real[:, 1],
})
comparison_df.round(1)


---

## 7. Elegir un umbral y señalar anomalías

Puntúa **todas** las travesías (no solo el conjunto de validación reservado) por su error de reconstrucción individual:

In [ ]:
autoencoder.eval()
with torch.no_grad():
    full_reconstruction = autoencoder(X_full_t)
    reconstruction_error = ((full_reconstruction - X_full_t) ** 2).mean(dim=1).numpy()

fuel["reconstruction_error"] = reconstruction_error
fuel["reconstruction_error"].describe()

No existe un umbral "correcto" universal — es una decisión de política, que compensa cuántas travesías se marcan para revisión frente a cuántas anomalías genuinas estás dispuesto a pasar por alto (la misma disyuntiva entre falsos positivos y falsos negativos que se discutió en el §9 de `NB13`, ahora sin etiquetas con las que medirla con precisión). Un punto de partida habitual y defendible: marcar el pequeño porcentaje superior por error:

In [ ]:
threshold = np.percentile(reconstruction_error, 95)
fuel["is_anomaly"] = fuel["reconstruction_error"] > threshold

print(f"Threshold (95th percentile): {threshold:.3f}")
print(f"Flagged as anomalous: {fuel['is_anomaly'].sum()} / {len(fuel)} voyages")

Visualiza la distribución del error y dónde cae el umbral:

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(reconstruction_error, bins=40)
plt.axvline(threshold, color="red", linestyle="--", label="95th percentile threshold")
plt.xlabel("Reconstruction error")
plt.ylabel("Number of voyages")
plt.title("Reconstruction error distribution")
plt.legend()
plt.show()

**Pruébalo tú mismo**: el error de reconstrucción de arriba promedia las 3 características — desglósalo por característica para las travesías peor reconstruidas. ¿Suele ser una característica la responsable, o contribuyen las tres aproximadamente por igual?

In [ ]:
with torch.no_grad():
    per_feature_error = ((full_reconstruction - X_full_t) ** 2).numpy()

top_anomaly_idx = np.argsort(reconstruction_error)[-5:][::-1]
pd.DataFrame(per_feature_error[top_anomaly_idx], columns=feature_cols).round(3)


---

## 8. Interpretar las anomalías frente a campos conocidos

Exactamente el mismo hábito de comprobación de sensatez de la clase de clustering de `NB09`: marcamos las anomalías usando **solo** las 3 características numéricas, sin tocar en ningún momento `weather_conditions` ni `ship_type`. ¿Las travesías marcadas coinciden con alguno de los dos, o el autoencoder encontró algo completamente distinto?

In [ ]:
pd.crosstab(fuel["is_anomaly"], fuel["weather_conditions"])

Y por tipo de buque:

In [ ]:
pd.crosstab(fuel["is_anomaly"], fuel["ship_type"])

**Lee tus propias tablas**: si las anomalías se concentran en tiempo `Stormy`, ese es un resultado tranquilizador y físicamente sensato — unas condiciones inusuales producen un comportamiento de combustible inusual, y el autoencoder lo encontró sin que se le dijera nada sobre el tiempo. Si en cambio las anomalías se reparten de forma uniforme entre tiempo y tipo de buque, eso es igual de informativo — sugiere que las travesías marcadas son inusuales por alguna otra razón (una ruta concreta, un problema de mantenimiento, un error de introducción de datos) que merece investigarse una por una, no un patrón que esta tabla cruzada por sí sola pueda explicar. Un diagrama de dispersión añade una comprobación visual de sentido común:

In [ ]:
plt.figure(figsize=(7, 5))
normal = fuel[~fuel["is_anomaly"]]
anomalous = fuel[fuel["is_anomaly"]]
plt.scatter(normal["distance"], normal["fuel_consumption"], alpha=0.4, label="Normal")
plt.scatter(anomalous["distance"], anomalous["fuel_consumption"], color="red", label="Flagged anomaly")
plt.xlabel("Distance (nm)")
plt.ylabel("Fuel consumption (L)")
plt.title("Flagged anomalies in distance/fuel-consumption space")
plt.legend()
plt.show()

**Pruébalo tú mismo**: las tablas cruzadas y los diagramas de dispersión muestran patrones agregados — mira directamente las 5 travesías más anómalas reales. ¿Sus valores reales de las características explican, en términos sencillos, por qué al autoencoder le costó reconstruirlas?

In [ ]:
top5 = fuel.iloc[top_anomaly_idx][["ship_id", "ship_type", "weather_conditions"] + feature_cols + ["reconstruction_error"]]
top5


---

## 9. Autoencoders frente a PCA, y más allá de la detección de anomalías

`NB09` usó PCA para comprimir los datos de Sonar, de 60 dimensiones, a 2 componentes para visualizarlos. El cuello de botella de un autoencoder hace algo estructuralmente similar — comprimir y después reconstruir — pero con una diferencia clave: PCA está restringido a proyecciones **lineales**, mientras que las capas `ReLU` de un autoencoder le permiten aprender compresión **no lineal**. En datos con estructura genuinamente no lineal, `un autoencoder puede capturar patrones que las proyecciones rectas de PCA no pueden`; en datos que son casi lineales, ambos suelen rendir de forma parecida, y PCA es mucho más simple de entrenar e interpretar.

La detección de anomalías es solo una aplicación. La misma idea de comprimir-y-reconstruir también sustenta:

| Aplicación | Cómo se usa el cuello de botella |
|---|---|
| Detección de anomalías (hoy) | Un error de reconstrucción alto marca ejemplos inusuales |
| Eliminación de ruido (denoising) | Se entrena con entrada ruidosa y objetivo limpio; el cuello de botella aprende a descartar el ruido |
| Reducción de dimensionalidad | Usar los propios valores del cuello de botella como características comprimidas para otro modelo |
| Preentrenamiento | Entrenar un codificador de forma no supervisada sobre abundantes datos sin etiquetar, y reutilizarlo — conceptualmente relacionado con el transfer learning de `NB15`, pero el "preentrenamiento" ocurre sobre tus propios datos sin etiquetar en vez de sobre el ImageNet etiquetado de otra persona |

**Pruébalo tú mismo**: pon a prueba directamente la afirmación lineal-frente-a-no-lineal. Ajusta un PCA de 2 componentes sobre los mismos datos escalados, reconstrúyelo (`pca.inverse_transform`), y compara su error de reconstrucción medio con el del autoencoder — ¿de verdad rinde el PCA de forma parecida aquí, o la no linealidad del autoencoder realmente ayuda en estos datos?

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_pca_reconstructed = pca.inverse_transform(X_pca)
pca_reconstruction_error = ((X_pca_reconstructed - X_scaled) ** 2).mean(axis=1)

print(f"Autoencoder mean reconstruction error: {reconstruction_error.mean():.4f}")
print(f"PCA (2 components) mean reconstruction error: {pca_reconstruction_error.mean():.4f}")


Comprueba cuál gana en tu ejecución. Si el PCA sale ganando (o muy cerca), eso es coherente con que las relaciones de estos datos sean casi lineales — exactamente el caso que describe el texto de esta sección, donde la simplicidad de PCA es la mejor opción práctica pese a la flexibilidad extra del autoencoder.

---

## Resumen de la clase

- Un autoencoder se entrena para reconstruir su propia entrada a través de un cuello de botella estrecho — sin etiquetas, la entrada es su propio objetivo.
- El cuello de botella fuerza la compresión; lo que la red no consigue comprimir bien (patrones raros, inusuales) se reconstruye mal.
- El error de reconstrucción se convierte en una puntuación de anomalía sin necesitar ninguna anomalía etiquetada — una ventaja práctica real frente a todos los clasificadores supervisados de este bloque.
- Entrenamos un autoencoder real sobre datos reales de combustible de buques, elegimos un umbral, marcamos travesías anómalas, y comprobamos si las marcas coincidían con el tiempo o el tipo de buque.
- Los autoencoders generalizan la idea de compresión de PCA a relaciones no lineales, y la misma arquitectura sustenta la eliminación de ruido, la reducción de dimensionalidad y el preentrenamiento autosupervisado.

## Para la próxima clase (NB17)

Cerramos el Bloque 3 con un repaso de todas sus arquitecturas — MLP, CNN, RNN/LSTM, transfer learning, autoencoders — y una discusión sobre cómo elegir entre ellas para un problema nuevo, nunca visto.

## Tarea / Ideas de práctica

1. Cambia `bottleneck_size` de 2 a 1 y a 3 — ¿cómo cambia la pérdida de reconstrucción de entrenamiento? ¿Qué implica un cuello de botella de 1 sobre cuánto puede representar realmente el modelo?
2. Cambia el umbral de anomalía del percentil 95 al 99 — ¿cuántas travesías se marcan ahora, y cuenta la tabla cruzada de tiempo/tipo de buque de la Parte 8 una historia distinta?
3. Vuelve a añadir `CO2_emissions` a `feature_cols` pese al aviso de fuga de datos de `NB07` — ¿cambia mucho el error de reconstrucción del autoencoder? Dado lo correlacionada que está con `fuel_consumption`, ¿lo esperarías?
4. Compara las travesías marcadas aquí con las asignaciones de cluster de `NB09` (si todavía conservas la salida de aquel notebook) — ¿tienden las anomalías a caer en un cluster concreto, o se reparten entre todos?
5. Explica con tus propias palabras por qué un autoencoder entrenado con datos mayoritariamente normales puede detectar anomalías que nunca vio explícitamente etiquetadas como tales — ¿qué está optimizando exactamente, y por qué eso generaliza a marcar lo inusual?

> ***Como siempre: una marca no supervisada es un punto de partida para investigar, no un veredicto — la tabla cruzada y el diagrama de dispersión de la Parte 8 son la diferencia entre "lo dijo el modelo" y entender de verdad qué se marcó.***